In [1]:
import os
import base64
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pathlib import Path
import magic
import open_clip
from PIL import Image
import torch
import faiss
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer, CrossEncoder
import open_clip
from typing import List,Dict
import numpy as np
from __future__ import annotations
from word_handler import Chunk, DocMeta, process_docx
from functools import lru_cache
import pickle

# Force Hugging Face to look directly at your D drive directory bypassing the link
os.environ["HF_HOME"] = r"D:\models\huggingface"
os.environ["TORCH_HOME"] = r"D:\models\torch_models"



C:\Users\shahin\AppData\Local\Temp\ipykernel_5304\979659067.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
load_dotenv()

hf_api_key = os.getenv("HUGGIN_FACE_API")



text_index    = None
text_meta: Dict[int, Chunk] = {}


image_index   = None
image_meta: Dict[int, dict] = {}


doc_meta_store:  Dict[str, DocMeta]        = {}
image_to_chunks: Dict[tuple[str, int], List[Chunk]]      = {}
table_to_chunks: Dict[tuple[str, int], List[Chunk]]      = {}
all_chunks:      List[Chunk] = []



BLEND_LOW_WEIGHTS  = (0.8, 0.2)
BLEND_HIGH_WEIGHTS = (0.2, 0.8)

IGNORE_IMAGE_THRESHOLD = 0.1
HIGH_IMAGE_THRESHOLD   = 0.4


STRUCTURAL_TERMS = {
    # Persian
    "بخش", "فصل", "مرحله", "گام",
    "اول", "اولین", "آخر", "آخرین",
    "پایانی", "نتیجه", "نتیجه گیری",
    "جمع بندی", "مقدمه",
    # English
    "chapter", "section", "part", "step",
    "first", "last", "final", "conclusion",
    "summary", "introduction",
}


In [3]:
client = OpenAI(
    api_key=hf_api_key,
    base_url="https://router.huggingface.co/v1"
)

model_name = "Qwen/Qwen3-4B-Instruct-2507"
vision_model_name = "Qwen/Qwen3-VL-8B-Instruct"

# Convert local image file to base64 string
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")




In [ ]:
##test#######
response = client.chat.completions.create(
    model= model_name,
    messages= [
        {
            "role":"user", "content":"give me a simple code for rag using langchain"
        }
    ]
)

# print(response.choices[0].message.content)
display(Markdown(response.choices[0].message.content))



local_image_path = r"C:\Users\shahin\Desktop\pics\er.jfif"
base64_image = encode_image_to_base64(local_image_path)
image_data_url = f"data:image/jpeg;base64,{base64_image}"

# Request execution block
response = client.chat.completions.create(
    model=vision_model_name,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what you see in this picture in detail."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data_url  # Passes the parsed base64 data string variable
                    }
                }
            ]
        }
    ]
)

display(Markdown(response.choices[0].message.content))

In [5]:


file_path =  r"F:\university\az e riz\گزارش.docx"

path = Path(file_path)

suffix, mime = None, None

if path.exists():
    suffix = path.suffix
    mime = magic.from_file(str(path), mime=True)


def route_file(mime, suffix, path):
    if mime == "application/pdf":
        #pdf
        handle_pdf(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        "application/msword"
    ]:
        #word
        handle_word(path)


    if mime in [
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        "application/vnd.ms-excel"
    ]:
        handle_excel(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        "application/vnd.ms-powerpoint"
    ]:

        handle_pp(path)

    # if mime.startswith("image/"):
    #     return "Image"
    #
    # if mime.startswith("audio/"):
    #     return "Audio"
    #
    # if mime.startswith("video/"):
    #     return "Video"

    return "invalid"







###########Alternative using model###################
# print(mime, suffix)
# system_prompt = """
# Your role is just to analyse the MIME and the suffix passed to you and detect the file type.
# You must identify if it is Excel, Word document, Powerpoint, Audio, Video, Image or PDF.
# Just return one word.
# If what is provided to you is not valid just return the word: invalid.
# """
#
# response = client.chat.completions.create(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     messages=[
#         {"role": "system", "content": system_prompt},
#         {
#             "role": "user",
#             "content": f"MIME: {mime}, suffix: {suffix}"
#         }
#     ]
# )
#
# print(response.choices[0].message.content)

AttributeError: module 'magic' has no attribute 'from_file'

In [4]:
from word_handler import process_docx

indexed_chunks: set = set()
indexed_images: set = set()
indexed_docs:   set = set()


def ingest(chunks: List[Chunk], doc_meta: DocMeta,
           img_to_ch: dict, tbl_to_ch: dict):
    doc_meta_store[doc_meta.doc_id] = doc_meta
    all_chunks.extend(chunks)
    for k, v in img_to_ch.items():
        image_to_chunks.setdefault(k, []).extend(v)
    for k, v in tbl_to_ch.items():
        table_to_chunks.setdefault(k, []).extend(v)

def handle_word(path: str):
    chunks, doc_meta, img_to_ch, tbl_to_ch = process_docx(path)

    if doc_meta.doc_id in indexed_docs:
        print(f"⏭️  skipped {doc_meta.doc_id} (already indexed)")
        return

    ingest(chunks, doc_meta, img_to_ch, tbl_to_ch)
    index_chunks(chunks)
    index_images(chunks)
    indexed_docs.add(doc_meta.doc_id)

def handle_excel(path):
    pass

def handle_pp(path):
    pass


def handle_pdf(path):
    pass

**Embeddings models**


1. mulitlangual e5-v2 for sentences
2. open-clip for images


In [5]:
from sentence_transformers import SentenceTransformer

e5_model = SentenceTransformer("intfloat/multilingual-e5-base")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="laion2b_s34b_b79k"
)
reranker = CrossEncoder("BAAI/bge-reranker-base", trust_remote_code=True)

clip_model.eval()


def embed_text(texts: List[str], is_query: bool = False) -> np.ndarray:

    prefix  = "query: " if is_query else "passage: "
    prefixed = [prefix + t for t in texts]
    vecs = e5_model.encode(prefixed, normalize_embeddings=True)
    return np.array(vecs, dtype="float32")

def embed_image(image_paths: List[str]) -> np.ndarray:
    images = torch.stack([
        clip_preprocess(Image.open(p).convert("RGB"))
        for p in image_paths
    ])                                              # shape (n, 3, 224, 224)

    with torch.no_grad():
        vecs = clip_model.encode_image(images)     # shape (n, 512)

    vecs = vecs /np.linalg.norm(vecs, keepdims=True)
    return vecs.cpu().numpy().astype("float32")


**Faiss Docstores**

In [7]:
def index_chunks(chunks: List[Chunk]):
    """Embed chunk texts and insert into the text FAISS index."""
    global text_index


    new_chunks = [
        c for c in chunks
        if (c.doc_id, c.chunk_index) not in indexed_chunks
    ]
    if not new_chunks:
        return

    texts = [c.text for c in new_chunks]
    vecs  = embed_text(texts, is_query=False)       # (n, 768)

    if text_index is None:
        text_index = faiss.IndexFlatIP(vecs.shape[1])

    base = text_index.ntotal
    text_index.add(vecs)
    for i, chunk in enumerate(new_chunks):
        text_meta[base + i] = chunk
        indexed_chunks.add((chunk.doc_id, chunk.chunk_index))


def index_images(chunks: List[Chunk]):
    """Embed images from image_context chunks and insert into the image FAISS index."""
    global image_index

    paths, records = [], []
    for c in chunks:
        if not c.image_refs:
            continue
        for img_id, path in c.image_refs.items():
            if (c.doc_id, img_id) in indexed_images:
                continue
            if os.path.exists(path):
                paths.append(path)
                records.append({
                    "image_id":    img_id,
                    "path":        path,
                    "doc_id":      c.doc_id,
                    "chunk_index": c.chunk_index,
                    "section_path": c.section_path,
                })

    if not paths:
        return

    vecs = embed_image(paths)                       # (m, 512)

    if image_index is None:
        image_index = faiss.IndexFlatIP(vecs.shape[1])

    base = image_index.ntotal
    image_index.add(vecs)
    for i, rec in enumerate(records):
        image_meta[base + i] = rec
        indexed_images.add((rec["doc_id"], rec["image_id"]))

**retreival**

In [8]:

def search_text(query: str, k: int = 5, oversample_factor: int = 4) -> List[dict]:
    if text_index is None or text_index.ntotal == 0:
        return []

    q_vec = embed_text([query], is_query=True)  #


    broad_k = min(k * oversample_factor, text_index.ntotal)
    scores, ids = text_index.search(q_vec, broad_k)

    results = []
    seen_texts = set()

    for score, fid in zip(scores[0], ids[0]):
        if fid == -1:
            continue
        chunk = text_meta[fid]


        if chunk.text in seen_texts:
            continue
        seen_texts.add(chunk.text)

        results.append({
            "faiss_score":  round(float(score), 4),
            "text":         chunk.text,
            "doc_id":       chunk.doc_id,
            "chunk_index":  chunk.chunk_index,
            "chunk_type":   chunk.chunk_type,
            "section_path": chunk.section_path,
            "image_refs":   chunk.image_refs,
            "table_refs":   chunk.table_refs,
        })


    return results


def search_image(query: str, k: int = 5) -> List[dict]:
    if image_index is None or image_index.ntotal == 0:
        return []
    q_token = open_clip.tokenize([query])

    with torch.no_grad():
        q_vec = clip_model.encode_text(q_token)
    q_vec = q_vec/np.linalg.norm(q_vec, keepdims=True)
    q_vec = q_vec.cpu().numpy().astype("float32")

    scores, ids = image_index.search(q_vec, k)

    results = []
    for score, fid in zip(scores[0], ids[0]):
        if fid == -1:
            continue
        meta = image_meta[fid]
        results.append({
            "score":        round(float(score), 4),
            "image_id":     meta["image_id"],
            "path":         meta["path"],
            "doc_id":       meta["doc_id"],
            "chunk_index":  meta["chunk_index"],
            "section_path": meta["section_path"],
        })
    return results



def build_context(text_results:List[dict], image_results:List[dict]) ->str:
    parts = []

    for r in text_results:
        path = " > ".join(r["section_path"]) if r["section_path"] else r["doc_id"]
        parts.append(f"[{path}]\n{r['text']}")
    if image_results:
        for r in image_results:
              key = (r["doc_id"], r["image_id"])
              chunks  = image_to_chunks.get(key,[])
              if chunks:
                  related = chunks[0]
                  path=">".join(r["section_path"] if r["section_path"] else r["doc_id"])
                  parts.append(f"[{path}| image_{r["image_id"]}]\n context: {related.text}")

    return "\n___\n".join(parts)

def rerank(query: str, chunks: List[dict],
           top_k: int = 5, threshold: float = 0.1) -> List[dict]:
    if not chunks:
        return []

    pairs = [(query, chunk["text"]) for chunk in chunks]
    rerank_scores = reranker.predict(pairs)

    reranked = []
    for chunk, score in zip(chunks, rerank_scores):
        result = dict(chunk)
        result["score"] = round(float(score), 4)
        reranked.append(result)

    reranked.sort(key=lambda x: x["score"], reverse=True)
    return [r for r in reranked if r["score"] > threshold][:top_k]

In [9]:
@lru_cache(maxsize=256)
def improve_query(query: str) -> str:
    """
    Cached — same query string won't trigger a second LLM call
    within the same session.
    """
    system_prompt = f"""
        You are a query expansion assistant for a multilingual RAG system.

        Your goal is to improve retrieval recall by expanding the user's query with:
        - Synonyms
        - Alternative phrasings
        - Related technical terminology
        - Common document structure terms when appropriate
        - Sequential references (first, second, final, etc.) when appropriate

        Rules:
        1. Keep the original query.
        2. Preserve the original language. Never translate.
        3. Add only highly plausible alternatives.
        4. Do not invent facts or specific information.
        5. Do not answer the query.
        6. Do not explain your reasoning.
        7. Output ONLY a comma-separated list.
        8. Keep the expansion concise (typically 5-10 terms/phrases total).

        Examples:

        Query: آخرین بخش آزمایش
        Output: آخرین بخش آزمایش, بخش پایانی, بخش آخر, قسمت نهایی, مراحل نهایی, بخش دوم

        Query: نصب کتابخانه پایتون
        Output: نصب کتابخانه پایتون, نصب پکیج پایتون, افزودن کتابخانه, راه اندازی کتابخانه, نصب وابستگی

        Query: database connection error
        Output: database connection error, database connectivity issue, connection failure, database access problem

        Query: first step of installation
        Output: first step of installation, installation beginning, setup start, initial installation step, step one

        User Query: {query}

        Optimized Search Query (Output only the terms):
    """

    result = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "system", "content": system_prompt}]
    )
    return result.choices[0].message.content



def is_structural_query(query: str) -> bool:
    q = query.lower()
    return any(term.lower() in q for term in STRUCTURAL_TERMS)


def _chunk_to_dict(chunk) -> dict:
    return {
        "text":         chunk.text,
        "doc_id":       chunk.doc_id,
        "chunk_index":  chunk.chunk_index,
        "chunk_type":   chunk.chunk_type,
        "section_path": chunk.section_path,
        "image_refs":   chunk.image_refs,
        "table_refs":   chunk.table_refs,
    }


def retrieval_text(query: str, k: int = 3,
                   min_threshold: float = 0.08) -> List[dict]:

    primary_results  = search_text(query, k=2, oversample_factor=4)
    reranked_results = rerank(query=query, chunks=primary_results, threshold=-50)

    if reranked_results and reranked_results[0]["score"] >= min_threshold:
        print("✅ High confidence results matched!")
        return reranked_results

    print("⚠️  Low confidence — activating query expansion fallback...")
    new_query = improve_query(query)
    print(f"   Expanded query: '{new_query}'")

    fallback_candidates = search_text(new_query, k=k, oversample_factor=4)
    final_res = rerank(query=new_query, chunks=fallback_candidates,
                  top_k=k, threshold=0.0)

    return final_res


def _search_and_rerank_images(query: str, rerank_query: str) -> tuple[List[dict], dict]:

    image_results = search_image(query, k=5)

    image_scores = {
        (r["doc_id"], r["chunk_index"]): r["score"]
        for r in image_results
    }

    candidate_chunks = [
        _chunk_to_dict(chunk)
        for chunk in all_chunks
        if (chunk.doc_id, chunk.chunk_index) in image_scores
    ]

    reranked = rerank(query=rerank_query, chunks=candidate_chunks)
    return reranked, image_scores

def retrieval_image(query: str, k: int = 3,
                    min_threshold: float = 0.08) -> List[dict]:

    reranked, image_scores = _search_and_rerank_images(query, rerank_query=query)


    if not reranked or reranked[0]["score"] < min_threshold:
        print("⚠️  Low image confidence — activating query expansion fallback...")
        expanded_query          = improve_query(query)
        reranked, image_scores  = _search_and_rerank_images(
            expanded_query, rerank_query=expanded_query
        )

    structural = is_structural_query(query)
    rw, iw_low, iw_high = 1.0, BLEND_LOW_WEIGHTS, BLEND_HIGH_WEIGHTS

    final_results = []
    for r in reranked:
        key          = (r["doc_id"], r["chunk_index"])
        image_score  = image_scores.get(key, 0.0)
        rerank_score = r["score"]

        if structural or image_score < IGNORE_IMAGE_THRESHOLD:
            final_score = rerank_score
        elif image_score < HIGH_IMAGE_THRESHOLD:
            final_score = iw_low[0] * rerank_score + iw_low[1] * image_score
        else:
            final_score = iw_high[0] * rerank_score + iw_high[1] * image_score

        result = dict(r)
        result["image_score"] = image_score
        result["final_score"] = round(final_score, 4)
        final_results.append(result)
    print(final_results)
    final_results.sort(key=lambda x: x["final_score"], reverse=True)
    return final_results[:k]

def retrieval(query: str, k: int = 3,
              min_threshold: float = 0.08):

    text_results  = retrieval_text(query, k, min_threshold)
    image_results = retrieval_image(query, k, min_threshold)
    for r in image_results:
        print(r)
        print("\n\n")

    #
    # seen    = set()
    # combined: List[dict] = []
    #
    # for r in text_results + image_results:
    #     key = (r["doc_id"], r["chunk_index"])
    #     if key not in seen:
    #         seen.add(key)
    #         combined.append(r)
    #
    # context = build_context(
    #     text_results  = [r for r in combined if r in text_results],
    #     image_results = [r for r in combined if r in image_results],
    # )
    #
    # return context




**Save the vectorstore**

In [ ]:

def save(directory: str):
    os.makedirs(directory, exist_ok=True)
    if text_index:
        faiss.write_index(text_index,  os.path.join(directory, "text.index"))
    if image_index:
        faiss.write_index(image_index, os.path.join(directory, "image.index"))
    with open(os.path.join(directory, "meta.pkl"), "wb") as f:
        pickle.dump({
            "text_meta":       text_meta,
            "image_meta":      image_meta,
            "doc_meta_store":  doc_meta_store,
            "image_to_chunks": image_to_chunks,
            "table_to_chunks": table_to_chunks,
            "all_chunks":      all_chunks,
            "indexed_chunks":  indexed_chunks,
            "indexed_images":  indexed_images,
            "indexed_docs":    indexed_docs,
        }, f)
    print(f"saved to {directory}/")


def load(directory: str):
    global text_index, image_index

    tp = os.path.join(directory, "text.index")
    ip = os.path.join(directory, "image.index")
    mp = os.path.join(directory, "meta.pkl")

    if os.path.exists(tp):  text_index  = faiss.read_index(tp)
    if os.path.exists(ip):  image_index = faiss.read_index(ip)
    if os.path.exists(mp):
        with open(mp, "rb") as f:
            m = pickle.load(f)
        text_meta.update(m["text_meta"])
        image_meta.update(m["image_meta"])
        doc_meta_store.update(m["doc_meta_store"])
        image_to_chunks.update(m["image_to_chunks"])
        table_to_chunks.update(m["table_to_chunks"])
        all_chunks.extend(m["all_chunks"])
        indexed_chunks.update(m.get("indexed_chunks", set()))
        indexed_images.update(m.get("indexed_images", set()))
        indexed_docs.update(m.get("indexed_docs",   set()))
    print(f" loaded from {directory}/")



In [10]:
if __name__ == "__main__":
    handle_word(r"F:\university\az e riz\گزارش.docx")

    results = retrieval(query="آخرین بخش آزمایش")
    print("final results")
    # for r in results:
    #     print(r)
    #context = build_context(results, [])
    #print(context[:600])


⚠️  Low confidence — activating query expansion fallback...
   Expanded query: 'آخرین بخش آزمایش, بخش پایانی, بخش آخر, قسمت نهایی, مراحل نهایی, بخش دوم'
⚠️  Low image confidence — activating query expansion fallback...
[{'text': 'بخش دوم: روشن شدن شدن همه نمایشگر ها و شمارش از 2000 تا 0: ایده اصلی این است که در یک لحظه همزمان همه سون سگمت ها روشن باشند و یک عدد را نشان دهند ولی در هر لحظه با استفاده از portB فقط یکی فعال باشد و بتواند تغییر کند. سپس یک تاخیر بسیار بسیار کوچک که چشم ما نتواند آنرا تشخیص دهد ایجاد کرده ، آن سون سگمت را غیرفعال و سون سگمت سمت راستش را فعال کرده و به آن اجازه تغییر میدهیم. به این ترتیب به بیننده ایده تغییر همزمان سون سگمت ها را میدهیم. برای اینکار فقط کافی است حلقه while   را تغییر دهیم. بدین ترتیب که در یک حلقه از 2000 تا 0 ، ارقام یکان دهگان صدگان و هزارگان عدد را استخراج می\u200cکنیم.(با استفاده از تقسیم و باقی مانده گیری بر 10 ،100، 1000).  حال ابتدا 0x08 را در پورت B  میریزیم تا چپ ترین سون سگمت فعال شود.  حال display_digit را   برای   رقم هزارگان صدا